# Train the Khmer OCR recognizer

Works unmodified on **Colab**, **Kaggle**, and a **local machine** -- on **CPU, GPU, or TPU**:
- Colab: mounts Google Drive at `/content/drive/My Drive/tuna-ocr` and checkpoints there.
- Kaggle: checkpoints to `/kaggle/working/tuna-ocr` (persisted as notebook output).
- Local: checkpoints to `recognizer/checkpoints/` in the repo.

The accelerator (TPU / GPU / CPU) is auto-detected -- no notebook changes needed either
way. **To actually get a TPU**, select it as the runtime/accelerator in Colab
(Runtime > Change runtime type > TPU) or Kaggle (Settings > Accelerator > TPU) *before*
running this notebook; both platforms ship `torch_xla` preinstalled on their TPU
runtimes, so no extra install step is needed here. On GPU, batch size is auto-probed
to fit the available VRAM (OOM-probing auto-tune); on TPU that probe is skipped (XLA
doesn't surface a catchable Python OOM the same way) and the configured batch size is
used as-is.

This pulls a **real, at-scale dataset** by default (all of `deepcopy_khmer_text_recognition`
/ `darayut_scene_text` / `sokheng_synthetic_v1`, plus 100k rows of
`chanrith_ocr_image_line` -- see section 2), roughly ~9GB on disk -- this is a
production-scale training run, not the small notebook-scale loop in
`notebooks/train_diagnose_eval.ipynb`. Use that other notebook first if you just want
to iterate quickly on model/training-loop changes.

Training logs every 100 steps and pushes a checkpoint to the `Panhapich/tuna-ocr`
Hugging Face repo (created private) every 10,000 steps.

**Before running:** add an `HF_TOKEN` secret (Colab: key icon in the left sidebar;
Kaggle: Add-ons > Secrets; local: `export HF_TOKEN=hf_...`).

**TPU caveat:** PyTorch/XLA's support for the CTC loss op has historically been
inconsistent across versions -- if training errors out or looks unusually slow on TPU,
check whether `torch.nn.functional.ctc_loss` is silently falling back to a CPU path
before assuming it's a bug in this repo.

**Platform settings to check first:**
- **Kaggle**: internet access is off by default -- Settings (right sidebar) > Internet >
  On. Without it, both `pip install` and the data-pull cell (which streams from the
  Hugging Face Hub) will fail. Also set Settings > Accelerator to GPU/TPU if you want
  one. Kaggle sessions have a disk budget too (check Settings) -- this notebook's
  ~9GB default pull should fit comfortably, but reduce `SAMPLES_PER_SOURCE` (section 2)
  if you're tight on space.
- **Colab**: Runtime > Change runtime type > pick GPU or TPU if you want one (default
  is CPU-only).


In [ ]:
import os, subprocess, sys

def detect_environment():
    # Kaggle is checked first: some Kaggle kernels leak a stray COLAB_GPU/
    # COLAB_RELEASE_TAG env var, which would otherwise misdetect as Colab and
    # crash trying to mount Google Drive. "/kaggle/working" existing is a much
    # harder signal to spoof than an env var, so it takes priority.
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.isdir("/kaggle/working"):
        return "kaggle"
    if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ:
        return "colab"
    return "local"

ENV = detect_environment()
print("environment:", ENV)

REPO_URL = "https://github.com/Pich09/tuna-ocr.git"

if not os.path.isdir("recognizer"):
    subprocess.run(["git", "clone", REPO_URL, "tuna-ocr"], check=True)
    os.chdir("tuna-ocr")

sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

In [ ]:
# Colab and Kaggle both ship torch preinstalled and matched to their runtime's CUDA
# driver -- blindly `pip install torch` on top of that (e.g. via a plain
# `-r recognizer/requirements.txt`) can silently replace it with a build that doesn't
# match, breaking GPU support. Install everything else normally, and only pip-install
# torch if it isn't importable at all (a bare local venv).
import importlib.util
from pathlib import Path

def strip_torch(req_path):
    lines = Path(req_path).read_text().splitlines()
    return [l for l in lines if not l.strip().lower().startswith("torch")]

reqs = strip_torch("recognizer/requirements.txt") + strip_torch("real_data/requirements.txt")
Path("/tmp/_notebook_requirements.txt").write_text("\n".join(reqs) + "\n")
!pip install -q -r /tmp/_notebook_requirements.txt

if importlib.util.find_spec("torch") is None:
    print("torch not found -- installing (no preinstalled build to preserve here)")
    !pip install -q torch
else:
    import torch
    print(f"using preinstalled torch {torch.__version__} (cuda available: {torch.cuda.is_available()}) -- not reinstalling")


In [ ]:
from recognizer import env_utils

checkpoint_root = env_utils.get_checkpoint_root(ENV)
hf_token = env_utils.get_hf_token(ENV)
accelerator = env_utils.detect_accelerator()
print("checkpoint root:", checkpoint_root)
print("HF token loaded:", bool(hf_token))
print("accelerator:", accelerator)
if accelerator == "cpu":
    print("no GPU/TPU detected -- training will be slow. On Colab: Runtime > Change "
          "runtime type. On Kaggle: Settings > Accelerator.")

In [ ]:
# Downloads Panhapich/khmer-sp-8k's SentencePiece model + khmer_segmentation.py
# wrapper (a bare .model file is not enough -- see recognizer/README.md).
from recognizer.tokenizer.fetch_tokenizer import fetch_tokenizer

fetch_tokenizer()

## 2. Data

The full pull -> pack -> dedup pipeline below is expensive (real network transfer +
CPU-bound hashing, potentially a long time at this scale) -- it only needs to run
**once**. The first successful run pushes its result to a private Hugging Face
dataset repo (`real_data.config.HF_DATA_REPO_ID`); every later run (new session, new
notebook, different machine) checks that repo first and just downloads the prebuilt
`dedup.arrow` instead of repeating the pull/pack/dedup work from scratch.

`SAMPLES_PER_SOURCE` only matters the first time (before anything's been pushed to the
Hub). The defaults pull everything available from the three smaller sources, but cap
`chanrith_ocr_image_line` (12M+ rows, ~40GB in full) at 100k rows. Each source is
pulled, packed into a single Arrow file (`<source>.arrow`, image bytes stored exactly
as pulled -- no re-encoding/resizing), and its raw per-image files are deleted before
the next source starts, bounding peak disk usage to "one source's raw files + all
Arrow files packed so far."


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
from real_data.config import EXTERNAL_DATASETS, HF_DATA_REPO_ID, REAL_DATA_ROOT
from real_data import hf_push

# Per-source sample counts for the (one-time) pull from source. Pulls everything
# available from the smaller sources, but caps chanrith_ocr_image_line (12M+ rows) at
# 100k -- pulling it in full would be ~40GB, far more than a Kaggle/Colab session's
# disk budget can hold.
SAMPLES_PER_SOURCE = {
    "deepcopy_khmer_text_recognition": 136_117,
    "chanrith_ocr_image_line": 100_000,
    "darayut_scene_text": 102_500,
    "sokheng_synthetic_v1": 100_000,
}

# KMP_DUPLICATE_LIB_OK/OMP_NUM_THREADS: Colab/Kaggle commonly have more than one
# OpenMP runtime on the import path (numpy, PIL/imagehash, datasets' native deps each
# bundle their own libomp/libiomp5) -- loading two in one process is a well-known cause
# of an immediate SIGABRT with zero output, right at import time, before any of this
# script's own code runs. Setting these before spawning avoids that class of crash;
# harmless if it wasn't actually the cause.
SUBPROCESS_ENV = {**os.environ, "KMP_DUPLICATE_LIB_OK": "TRUE", "OMP_NUM_THREADS": "1"}

def run_checked(cmd):
    """Runs `cmd`, always printing its output, and raises with the actual captured
    stderr on failure -- a bare `subprocess.CalledProcessError` (or, worse, a `!shell`
    cell whose exit code isn't checked at all) hides exactly the text that explains
    *why* it died, which is the difference between a one-line fix and a guessing game."""
    result = subprocess.run(cmd, env=SUBPROCESS_ENV, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        sys.stderr.write(result.stderr)
        raise RuntimeError(
            f"command failed (exit code {result.returncode}"
            f"{', likely killed by a signal -- see stderr above for the real cause' if result.returncode < 0 else ''}"
            f"): {' '.join(cmd)}"
        )
    return result

dedup_manifest = REAL_DATA_ROOT / "samples" / "dedup.arrow"
built_locally = False  # tracks whether THIS run built dedup_manifest from source
                        # (vs. it already being local, or downloaded from the Hub) --
                        # only push to the Hub in the first case (section 2b below).

if dedup_manifest.exists():
    print(f"{dedup_manifest} already present locally, skipping pull/download")
elif hf_push.dataset_exists_on_hub(HF_DATA_REPO_ID, token=hf_token):
    print(f"found a prebuilt dataset on the Hub ({HF_DATA_REPO_ID}) -- downloading "
          f"instead of re-pulling/re-deduplicating from source")
    hf_push.pull_dataset(dedup_manifest, token=hf_token, repo_id=HF_DATA_REPO_ID)
else:
    print(f"no prebuilt dataset found on {HF_DATA_REPO_ID} -- pulling + packing from "
          f"source (one-time cost; result gets pushed to the Hub in the next cell)")
    built_locally = True

    # Pull -> pack to Arrow -> delete raw, one source at a time (not all sources
    # pulled first, then packed): this bounds peak disk usage to "current source's
    # raw files + every Arrow file packed so far", instead of needing all 4 sources'
    # raw files on disk simultaneously.
    arrow_files = []
    for source in EXTERNAL_DATASETS:
        arrow_path = REAL_DATA_ROOT / "samples" / f"{source}.arrow"
        arrow_files.append(arrow_path)
        if arrow_path.exists():
            print(f"{source}: already packed, skipping")
            continue

        source_dir = REAL_DATA_ROOT / "samples" / source
        num_samples = SAMPLES_PER_SOURCE[source]
        if not (source_dir / "manifest.tsv").exists():
            print(f"{source}: pulling {num_samples} samples...")
            run_checked([sys.executable, "-m", "real_data.generate_external_chunks",
                         "--source", source, "--num-samples", str(num_samples)])

        print(f"{source}: packing to {arrow_path}...")
        run_checked([sys.executable, "-m", "real_data.pack_arrow",
                     "--source", source, "--delete-raw"])

    print(arrow_files)


In [ ]:
# 2b. Deduplicate + push to the Hub -- only runs if this session actually built the
# dataset from source above (built_locally == True); a no-op if dedup_manifest was
# already local or was just downloaded from the Hub.
if built_locally:
    # --near-dup-threshold 0 disables the O(n^2) near-dup pass -- REQUIRED at this
    # scale (hundreds of thousands of rows): the default pairwise comparison is
    # O(n^2) and would take an impractically long time (the nonzero default is only
    # tuned/safe for the notebook-scale hundreds-to-thousands range, e.g.
    # notebooks/train_diagnose_eval.ipynb).
    missing = [str(p) for p in arrow_files if not p.exists()]
    if missing:
        raise RuntimeError(
            "The following sources are missing their packed .arrow file -- re-run "
            "the pull cell above (in full, for all 4 sources) before deduplicating. "
            "This usually means the runtime restarted/reset between the pull and "
            "dedup cells (e.g. after a crash) and the previously-pulled data under "
            f"{REAL_DATA_ROOT} was lost:\n  " + "\n  ".join(missing)
        )

    run_checked([sys.executable, "-m", "real_data.deduplicate",
                 "--arrow-files", *[str(p) for p in arrow_files],
                 "--out", str(dedup_manifest),
                 "--near-dup-threshold", "0"])
    assert dedup_manifest.exists(), (
        f"{dedup_manifest} was not created -- check the pull cell above actually "
        f"populated {[str(p) for p in arrow_files]} before dedup ran."
    )
    print("dedup arrow file ready:", dedup_manifest)

    print(f"pushing prebuilt dataset to the Hub ({HF_DATA_REPO_ID}) so future runs "
          f"can skip straight to downloading it...")
    url = hf_push.push_dataset(dedup_manifest, token=hf_token, repo_id=HF_DATA_REPO_ID, private=True)
    print("pushed:", url)
else:
    print(f"{dedup_manifest} already ready (local or from the Hub) -- nothing to dedup/push")


In [ ]:
from pathlib import Path

from recognizer.config import ModelConfig, TrainConfig
from recognizer.train import run_training
from recognizer.hf_push import pull_latest_checkpoint

model_cfg = ModelConfig()
train_cfg = TrainConfig(
    num_workers=0,  # DataLoader workers fork *after* CUDA is initialized in a notebook
                    # kernel, which is unsafe and can deadlock silently (GPU pinned at
                    # 100% with zero real steps). 0 = safe synchronous loading.
    sequential_ar_steps=20_000,  # ~10% of max_steps=200_000 (default). Trains the AR
                                 # decoder in plain sequential mode first (full teacher
                                 # forcing, unrestricted cross-attention) so it learns
                                 # real image-to-text alignment before switching to
                                 # blockwise mode -- see recognizer/README.md's
                                 # "Blockwise-AR decoding" section for why: training
                                 # blockwise from step 0 was observed to make the AR
                                 # decoder fit the uniform block-split approximation
                                 # instead of the image, capping its accuracy well
                                 # below CTC's on the same encoder.
)  # log_every=100, ckpt_every=10_000 by default

RUN_NAME = "v1"
CHECKPOINT_REPO_ID = "Panhapich/tuna-ocr"

# False (current setting): always start from scratch, ignoring any checkpoint that
# already exists locally or on the Hub -- the new run's step_*.pt files overwrite the
# old ones of the same name in CHECKPOINT_REPO_ID as it goes. Set True to instead
# continue from where a previous session left off: it prefers a local
# checkpoints/<RUN_NAME>/last.pt (same session, merely interrupted), then falls back
# to the highest-step checkpoint pushed to the Hub (fresh Colab/Kaggle session, whose
# local disk was wiped), then starts fresh if neither exists.
RESUME_FROM_CHECKPOINT = False

resume_path = None
if RESUME_FROM_CHECKPOINT:
    local_last = Path(checkpoint_root) / RUN_NAME / "last.pt"
    if local_last.exists():
        resume_path = local_last
        print(f"resuming from local checkpoint: {resume_path}")
    else:
        resume_path = pull_latest_checkpoint(Path(checkpoint_root) / RUN_NAME, token=hf_token,
                                             repo_id=CHECKPOINT_REPO_ID)
        if resume_path:
            print(f"no local checkpoint -- resuming from the latest one on the Hub: {resume_path}")
        else:
            print(f"no local or Hub checkpoint found for {CHECKPOINT_REPO_ID} -- starting a fresh run")
else:
    print("RESUME_FROM_CHECKPOINT is False -- starting from scratch and overwriting "
          f"existing checkpoints in {CHECKPOINT_REPO_ID} as the run progresses")

# The Panhapich/tuna-ocr HF repo is created private by default the first time
# a checkpoint is pushed (hub_private=True) -- set to False only if you've
# deliberately decided the checkpoint repo should be public.
model = run_training(
    model_cfg, train_cfg,
    dedup_manifest_path=dedup_manifest,
    checkpoint_root=checkpoint_root,
    run_name=RUN_NAME,
    push_to_hub=True,
    repo_id=CHECKPOINT_REPO_ID,
    hf_token=hf_token,
    hub_private=True,
    auto_batch_size=True,  # OOM-probing auto-tune; no-op on CPU
    resume_path=resume_path,
)
